###### so the thing is we will be fetching the details from a website and will give the infor of that website to the llm to make the llm repsond to the quiestions user will ask regarding the website


In [45]:
# data ingestion-> data loading
from langchain_community.document_loaders import WebBaseLoader

loader=WebBaseLoader("https://www.warmerbakesncakes.com/menu")

data=loader.load()

In [46]:
data

[Document(metadata={'source': 'https://www.warmerbakesncakes.com/menu', 'title': 'Warmer Bakes & Cakes', 'description': 'Professional AC installation, repair, and maintenance services. 24/7 emergency support with certified technicians.', 'language': 'en'}, page_content='\n\n\n\n\nWarmer Bakes & Cakes\n\n\n\n\n\n\n\n\n\n\n\n\n\nHome\nWhat We Make\nGallery\n\n\nWarmer Bakes & Cakes\n\n\nMenu\nAdmin\n☰\n\n\n\n\n\n\n\n\n✕\nHome\nWhat We Make\nGallery\nMenu\nAdmin\n\n\n\n\n\n\nWarmer Bakes & Cakes\n\n    \nOur Bakery Menu\n34 handcrafted products\n\n\n\n🔍\n\n✕\n\n\nAll Items\nCakes\nFestive\nCookies\nBreads\nAppetizers\nBakes\nDesserts\nPickles\nDips\n\n\n\n\n\nCakes\n\n\nLemon Drizzle Cake\nChoose Size\n\n\n              500 gm\n            \n\n              1 kg\n            \n\n              1.5 kg\n            \n\n              2 kg\n            \n\n₹800\n\n\n−\n1\n+\n\n\n            🛒 Add to Cart\n          \n\n\n\n\n\n\nBreads\n\n\nHot Cross Buns\nPrice on request\n\n\n−\n1\n+\n\n\n  

In [47]:
# split the data into chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=50)

splitted_data=text_splitter.split_documents(data)

In [48]:
splitted_data

[Document(metadata={'source': 'https://www.warmerbakesncakes.com/menu', 'title': 'Warmer Bakes & Cakes', 'description': 'Professional AC installation, repair, and maintenance services. 24/7 emergency support with certified technicians.', 'language': 'en'}, page_content='Warmer Bakes & Cakes\n\n\n\n\n\n\n\n\n\n\n\n\n\nHome\nWhat We Make\nGallery\n\n\nWarmer Bakes & Cakes\n\n\nMenu\nAdmin\n☰\n\n\n\n\n\n\n\n\n✕\nHome\nWhat We Make\nGallery\nMenu\nAdmin\n\n\n\n\n\n\nWarmer Bakes & Cakes'),
 Document(metadata={'source': 'https://www.warmerbakesncakes.com/menu', 'title': 'Warmer Bakes & Cakes', 'description': 'Professional AC installation, repair, and maintenance services. 24/7 emergency support with certified technicians.', 'language': 'en'}, page_content='Warmer Bakes & Cakes\n\n    \nOur Bakery Menu\n34 handcrafted products\n\n\n\n🔍\n\n✕\n\n\nAll Items\nCakes\nFestive\nCookies\nBreads\nAppetizers\nBakes\nDesserts\nPickles\nDips\n\n\n\n\n\nCakes'),
 Document(metadata={'source': 'https://ww

In [49]:
# embeddings 
# convert the data into embeddings means vectors 

from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)
embeddings

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3120.45it/s]


HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
), model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [50]:
from langchain_community.vectorstores import FAISS

vector_store=FAISS.from_documents(
    splitted_data,
    embedding=embeddings
)

In [51]:
vector_store

In [ ]:
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# this create_stuff_documents_Chain is to put all the retrived vectors intoo one prompt
# means retriver can fetch many documents based upon similarity serarchh and this will
# basically combine or put those all documents inonce

In [53]:
import os
from dotenv import load_dotenv

load_dotenv()

# setting the api keys in enviornemnt
os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
os.environ['GROQ_API_KEY']=os.getenv("GROQ_API_KEY")

os.environ['LANGCHAIN_API_KEY']=os.getenv("LANGCHAIN_API_KEY")
os.environ['LANGCHAIN_TRACING_V2'] = 'true'

os.environ['LANGCHAIN_PROJECT']="RAG"

In [54]:
# create llm

from langchain_groq import ChatGroq

llm=ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [55]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a helpful assistant.
        Answer the user's question using only the provided context.

        Context:
        {context}"""
            ),
            (
                "human",
                "{input}"
            )
    ])

# document chain ->The Document Chain takes the documents found by the retriever and gives them to the LLM in a prompt.
# so the document info is passed in prompt context...

# will do take all the documentss retrivedd by retriver and put them all in the prompt as a contenxt

document_chain=create_stuff_documents_chain(llm,prompt)

In [56]:
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are a helpful assistant.\n        Answer the user's question using only the provided context.\n\n        Context:\n        {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'tex

In [57]:
# testing with giving context on own
# creating the document manually for rigth nwo

from langchain_core.documents import Document

document_chain.invoke({
    "input": "What real East Indian food do you make?",
    "context": [
        Document(
            page_content="""We are a small home kitchen that loves to cook and bake.
            Everything we make comes from old family recipes passed down through generations.
            We make cakes, pies, pickles, and real East Indian food like
            Bottle Masala and Vindaloo — all made fresh, just for you."""
        )
    ]
})

# now see here we dont need retrival chain why? becuasee here the context is directly given by us manually but that will not happen we 
# want rag to read the information from the site 
# means we want the llm to provide the context and the context should come from the similarity search 
# that is from retriver
# so thatswhy retrival chain

#So the Retriever's job is: Find the useful documents from your large collection.

'We make real East Indian dishes such as **Bottle\u202fMasala** and **Vindaloo**, all prepared fresh in our kitchen.'

In [58]:
vector_store

In [59]:
retriever=vector_store.as_retriever()

# so now we can invokde the vectore stro similarity search by just retriver.invoke


In [60]:
from langchain_classic.chains import create_retrieval_chain

retrival_chain=create_retrieval_chain(
    retriever,
    document_chain
)

In [61]:
# get the response fro the llm

response=retrival_chain.invoke({"input":"What real East Indian food do you make?"})

print(response)

{'input': 'What real East Indian food do you make?', 'context': [Document(id='c9fe6ed1-961b-4010-9da3-8d9311fd62a2', metadata={'source': 'https://www.warmerbakesncakes.com/menu', 'title': 'Warmer Bakes & Cakes', 'description': 'Professional AC installation, repair, and maintenance services. 24/7 emergency support with certified technicians.', 'language': 'en'}, page_content='Breads\n\n\nStar Bread\n₹1000\n\n\n−\n1\n+\n\n\n            🛒 Add to Cart\n          \n\n\n\n\n\n\nEast indian\n\n\nVindaloo (Pork & Buff)\nChoose Size\n\n\n              1 kg\n            \n\n₹1500\n\n\n−\n1\n+'), Document(id='0d4bfd1e-c1a7-4539-8142-9d55f0717ab5', metadata={'source': 'https://www.warmerbakesncakes.com/menu', 'title': 'Warmer Bakes & Cakes', 'description': 'Professional AC installation, repair, and maintenance services. 24/7 emergency support with certified technicians.', 'language': 'en'}, page_content='1 kg\n            \n\n₹2500\n\n\n−\n1\n+\n\n\n            🛒 Add to Cart\n          \n\n\n\n\n\

In [62]:
type(response)

dict

In [63]:
print(response["answer"])

Based on the menu, the real East Indian dishes we offer are:

- **Vindaloo (Pork & Buff)** – available in 1 kg (₹1500) or 1 kg (₹2500) portions.  
- **Butter Chicken** – available in 1 kg (₹1600) or 1 kg (₹1500) portions.  
- **East Indian Bottle Masala** – available in a 250 gm pack for ₹600.


In [69]:
input_text=input("enter what you want to ask?")
response=retrival_chain.invoke({"input":input_text})
print(response["answer"])

Warmer Bakes & Cakes offers a range of handcrafted items across several categories, including:

- **Bakes** – e.g., Mushroom & Spinach Pie (available in 1 kg for ₹980)  
- **Cakes**  
- **Festive treats** – e.g., Marzipan (size selectable)  
- **Cookies**  
- **Breads**  
- **Appetizers** – e.g., Skewers (grape, cheese, olives) in both veg and non‑veg versions (price on request)  
- **Desserts**  
- **Pickles**  
- **Dips**  

In total, the menu lists 34 handcrafted products spanning these categories.
